# 00 — Protocol and datasets

The pre-registered protocol, the case set and the provenance of every ground-truth label. Labels used in every notebook: **MEASURED** (a real provider or real run produced the number), **ESTIMATED** (derived here from measured data by a stated formula), **SIMULATED** (mock runner, no real model), **NOT_RUN**, **UNKNOWN** (the evidence does not say). Implementation and wiring are reported separately from evaluation: code that exists is not a measured capability.

In [1]:
import json
import platform
import random
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 11  # the pre-registered bootstrap seed; reused for every derived computation here
random.seed(SEED)
np.random.seed(SEED)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "EVIDENCE_MAP.md").exists())  # research/council-ablation
EV = ROOT / "evidence"
JEV = ROOT.parent / "jev" / "evidence"  # the JEV study's canonical copy in this repository
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 220)


def load(rel):
    return json.loads((EV / rel).read_text())


def jsonl(rel):
    return [json.loads(x) for x in (EV / rel).read_text().splitlines() if x.strip()]


def final_calls(rel):
    """The harness cache is append-only: a re-asked call (Amendments 1 and 2) appends a new row and the
    last row per (member, case) is the one analysed. Earlier rows stay in the file as recorded calls."""
    return pd.DataFrame(jsonl(rel)).drop_duplicates(["member", "case_id"], keep="last").reset_index(drop=True)


R = load("council/results.json")
REG = {r["id"]: r for r in load("reasoning_methods/reasoning_methods_registry.json")}
print("python", platform.python_version(), "| pandas", pd.__version__, "| numpy", np.__version__, "| seed", SEED)
print("evidence files:", sorted(str(p.relative_to(ROOT)) for p in EV.rglob("*") if p.is_file()))


python 3.12.12 | pandas 3.1.0 | numpy 2.5.4 | seed 11
evidence files: ['evidence/SOURCE_MANIFEST.json', 'evidence/council/calibration_by_member.csv', 'evidence/council/cases.csv', 'evidence/council/cases.json', 'evidence/council/conditions.csv', 'evidence/council/cost_latency_by_condition.csv', 'evidence/council/judge_calls.csv', 'evidence/council/judge_calls.jsonl', 'evidence/council/member_calls.csv', 'evidence/council/member_calls.jsonl', 'evidence/council/member_calls_budget300_archive.csv', 'evidence/council/member_calls_budget300_archive.jsonl', 'evidence/council/results.json', 'evidence/decision_methods/ablation-2026-10-20.json', 'evidence/notebook_runs.json', 'evidence/publication_scan.json', 'evidence/reasoning_methods/method_status_summary.csv', 'evidence/reasoning_methods/reasoning_methods_registry.json', 'evidence/reasoning_methods/reasoning_strategies_mock_benchmark.md', 'evidence/reasoning_methods/research_plans.json', 'evidence/replay_check.json']


In [2]:
#: Evaluation state per method, with the evidence that sets it. Explicit, not inferred from text.
EVALUATION = {
    "deterministic-baseline": "MEASURED",      # reference arm of the decision-methods ablation
    "single-strong-model": "MEASURED",         # condition A of the council benchmark
    "council": "MEASURED",                     # conditions B1-D, F (harness, not the product endpoint)
    "proposer-judge": "MEASURED",              # as condition E (judge over C) in the harness only
    "critic-veto": "NOT_RUN",
    "critic-selective-council": "NOT_RUN",
    "cot": "SIMULATED (mock runner)",
    "tot": "SIMULATED (mock runner)",
    "got": "SIMULATED (mock runner)",
    "self-consistency": "NOT_RUN",
    "react": "NOT_RUN",
    "reflexion": "NOT_RUN",
    "pot-evidence": "NOT_RUN",
    "mcts": "NOT_RUN",
    "jev": "MEASURED",                         # real provider calls on decision-corpus-v1
    "agentic-llm-decision": "MEASURED",        # decision-methods ablation, small local model
    "hermes-agent-patterns": "NOT_RUN",
}


def status_row(method_id):
    """Implementation / wiring / evaluation labels, derived from the registry snapshot only."""
    r = REG[method_id]
    implemented = "NOT_IMPLEMENTED" if r["wiring"] == "NOT_PRESENT" else "IMPLEMENTED"
    wired = "WIRED" if r["wiring"] in {"WIRED", "DECISION_AFFECTING", "OBSERVED", "EVALUATED"} else "NOT_WIRED"
    evaluation = EVALUATION[method_id]
    return {"method": r["name"], "implemented": implemented, "wired": wired, "authority": r["authority"],
            "evaluation": evaluation, "production": r["production"], "recommendation": r["recommendation"],
            "result": r["result"], "limitations": r["limitations"]}


## Protocol

Hypotheses H0–H3, conditions A–F, the calibration/evaluation split and the decision rules were committed before the first model call (`protocol/PREREGISTRATION.md`). Two amendments were recorded before any result was read (`protocol/AMENDMENTS.md`): the Claude 5.5 models accept only temperature 1, and a 300-token budget truncated 186 answers, which were re-asked with 1 024 tokens.

In [3]:
C = load("council/cases.json")
cases = pd.DataFrame(C["cases"])
print("cases:", len(cases), "| calibration:", (cases.split == "calibration").sum(), "| evaluation:", (cases.split == "evaluation").sum())
pd.crosstab(cases.family, cases.split, margins=True)

cases: 110 | calibration: 39 | evaluation: 71


split,calibration,evaluation,All
family,,,
fe_suggestion,3,4,7
feature_screening,18,35,53
high_consequence,2,4,6
missing_source,2,4,6
model_selection,3,5,8
policy_conflict,3,5,8
target_interpretation,3,5,8
validation_interpretation,3,5,8
xai_interpretation,2,4,6


In [4]:
pd.crosstab([cases.difficulty, cases.consequence], cases.split, margins=True)

split                    calibration  evaluation  All
difficulty  consequence                              
adversarial high                   7          14   21
            normal                 7          17   24
ambiguous   high                   4           4    8
            normal                 0           5    5
easy        high                   8          18   26
            normal                13          13   26
All                               39          71  110

## Ground-truth provenance

Every label comes from public dataset documentation or from a rule stated inside the case. No label is produced by a model. **No dataset rows are included in this package**: a case quotes a column name and its documented meaning only.

In [5]:
ctx = pd.DataFrame([{"context": k, **v} for k, v in C["contexts"].items()])
ctx[["context", "dataset", "target", "decision_about_people", "source"]]

,context,dataset,target,decision_about_people,source
0,paysim,PaySim mobile-money transaction simulation,isFraud — the transaction was made by a fraudulent agent,False,"PaySim data dictionary (Lopez-Rojas et al., Kaggle 'Synthetic Financial Datasets For Fraud Detection')"
1,bank,UCI Bank Marketing (Moro et al. 2014),y — the client subscribed a term deposit,False,UCI Bank Marketing attribute information
2,titanic,"Titanic passenger list (titanic3, Vanderbilt)",survived,False,titanic3 variable descriptions
3,lc,Lending Club accepted loans 2007–2015,loan_status = Charged Off,True,Lending Club LoanStats data dictionary
4,adult,UCI Adult census income,income > 50K,True,UCI Adult attribute information
5,german,Statlog German Credit,credit risk (good/bad),True,Statlog German Credit attribute documentation
6,diab,Diabetes 130-US hospitals 1999–2008 (Strack et al. 2014),readmitted < 30 days,False,UCI Diabetes 130-US feature list
7,telco,IBM Telco customer churn,Churn (left within the last month),False,IBM Telco churn sample dataset
8,retail,Retail loyalty customers (constructed),churn = no purchase during the 60 days after the snapshot date,False,constructed: label follows from the stated target definition
9,credit_c,Consumer loans (constructed),default = 90+ days past due within 12 months after origination,True,constructed: label follows from the stated target definition


In [6]:
cases.source.map(lambda s: "constructed (rule stated in case)" if str(s).startswith("constructed") else "public documentation").value_counts()

source
public documentation                 58
constructed (rule stated in case)    52
Name: count, dtype: int64

## Integrity of the copied protocol and harness

The SHA-256 of each copied file is compared with the manifest written when the package was built from the source repository.

In [7]:
import hashlib
man = load("SOURCE_MANIFEST.json")
pairs = {"scripts/research/council/cases.py": "harness/cases.py", "scripts/research/council/run.py": "harness/run.py",
         "docs/research/council-ablation/PREREGISTRATION.md": "protocol/PREREGISTRATION.md",
         "docs/research/council-ablation/AMENDMENTS.md": "protocol/AMENDMENTS.md",
         "docs/research/council-ablation/evidence/results.json": "evidence/council/results.json"}
rows = [{"source": s, "package": p, "match": hashlib.sha256((ROOT / p).read_bytes()).hexdigest() == man["sha256"][s]} for s, p in pairs.items()]
df = pd.DataFrame(rows); assert df.match.all(), df; df

,source,package,match
0,scripts/research/council/cases.py,harness/cases.py,True
1,scripts/research/council/run.py,harness/run.py,True
2,docs/research/council-ablation/PREREGISTRATION.md,protocol/PREREGISTRATION.md,True
3,docs/research/council-ablation/AMENDMENTS.md,protocol/AMENDMENTS.md,True
4,docs/research/council-ablation/evidence/results.json,evidence/council/results.json,True


In [8]:
fz = (ROOT / "protocol/PREREGISTRATION.md").read_text()
print(fz[fz.index("## Freeze record"):])

## Freeze record

Committed before any evaluation call (2026-11-04). SHA-256 at freeze:

* `scripts/research/council/cases.py` — 7c488dd95fba79656e5999b9b637a0a1fc587840a165ae58e8e4f8af77f1f0ef
* `scripts/research/council/run.py` — 79a5967271b3cdcf7bb654f23db52cfb1c3715ac924934f39dc3c9abf484c28b

Any later change to either file is listed in `AMENDMENTS.md` with its reason, before results are read.

